# Decision Curve Analysis (DCA)

## Overview
This notebook evaluates the clinical net benefit of risk prediction models trained for identifying bacterial bloodstream infections / ESBL cases on the untouched temporal test set.

Decision Curve Analysis (DCA) compares the net benefit of model-guided clinical decision-making against two standard default strategies across a spectrum of decision threshold probabilities ($p_t$):
1. **Treat All (`treat_all`)**: Assume every patient is high-risk and administer broad-spectrum treatment.
2. **Treat None (`treat_none`)**: Assume no patient is high-risk and withhold broad-spectrum treatment (Net Benefit = $0$).

## Net Benefit Formula
The net benefit $NB$ at a given threshold probability $p_t$ is defined as:

$$NB(p_t) = \frac{\text{True Positives}}{N} - \frac{\text{False Positives}}{N} \times \left( \frac{p_t}{1 - p_t} \right)$$

where $N$ is the total number of patients, and $\frac{p_t}{1 - p_t}$ represents the clinical odds ratio weighting the harm of an unnecessary intervention relative to the benefit of treating a true case.

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Setup and Environment Configuration

We import necessary data manipulation, visualization, and custom DCA evaluation functions from `icare_risk`. Configuration parameters (target column, prediction suffixes, and output directories) are dynamically loaded from the analysis YAML configuration.

In [9]:
from pathlib import Path
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from icare_risk.data.audit import load_admission_file
from icare_risk.evaluation.decision_curve import (
    evaluate_decision_curves,
    decision_curve_summary,
    plot_decision_curves,
)

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

prediction_path = (
    Path(config['outputs']['prediction_directory'])
    / 'admission_predictions.parquet'
)
predictions = load_admission_file(prediction_path)

target = config['columns']['target']
target_mapping = config.get(
    'target_mapping',
    config.get('analysis', {}).get('target_mapping', {}),
)
prediction_columns = [
    column for column in predictions.columns if column.endswith('_prob')
]

if not prediction_columns:
    raise ValueError('No *_prob columns found in the prediction table.')

test_mask = (
    predictions['split'].eq('test')
    if 'split' in predictions.columns else None
)

print(f'Prediction rows: {len(predictions):,}')
print(f'Models: {prediction_columns}')
print(f'Test rows: {int(test_mask.sum()):,}' if test_mask is not None else 'No split column found')

Prediction rows: 10,000
Models: ['lr_kim_components_prob', 'lr_all_score_values_prob', 'lr_clinical_baseline_prob', 'xgb_kim_components_prob', 'xgb_all_score_values_prob', 'xgb_clinical_baseline_prob']
Test rows: 2,483


## 2. Decision Curve Evaluation

Thresholds ($p_t \in [0.01, 0.50]$) represent the predicted probability at which a clinician would initiate action (e.g., prescribing broad-spectrum antibiotics).

- **Low Thresholds ($p_t < 5\%$)**: Prioritize high sensitivity (avoiding missed ESBL cases), accepting more false positives.
- **Higher Thresholds ($p_t > 20\%$)**: Prioritize antimicrobial stewardship, minimizing unnecessary broad-spectrum exposure.

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

# Define thresholds and compute curves on the test set
thresholds = np.linspace(0.01, 0.50, 100)

# Compute decision curves.
curves = evaluate_decision_curves(
    predictions,
    target_column=target,
    prediction_columns=prediction_columns,
    thresholds=thresholds,
    subset=test_mask,
    target_mapping=target_mapping,
)

# Get summary
summary = decision_curve_summary(curves)
display(summary.head(10))

## 3. Exporting Results

Evaluation outputs are exported to CSV and PNG formats in the designated results directory:
- Raw curve values: `results/decision_curve/decision_curve_values.csv`
- Summary metrics: `results/decision_curve/decision_curve_summary.csv`

In [ ]:
# Define output directory
output_dir = Path(config['outputs']['decision_curve_directory'])
output_dir.mkdir(parents=True, exist_ok=True)

# Save raw curve values and summary metrics to CSV first
curves_csv_path = output_dir / "decision_curve_values.csv"
summary_csv_path = output_dir / "decision_curve_summary.csv"

curves.to_csv(curves_csv_path, index=False)
summary.to_csv(summary_csv_path, index=False)

print(f"Saved decision curve datasets to {output_dir}")

## 4. Loading Saved Data and Generating Plots

Load the saved `.csv` file back into memory to render the final decision curve plot.
- Plot figure: `results/decision_curve/decision_curve_plot.png`

In [ ]:
import matplotlib.pyplot as plt
from icare_risk.evaluation.decision_curve import plot_decision_curves

# Load the saved CSV data back into memory for plotting
loaded_curves = pd.read_csv(curves_csv_path)

# Set up a high-quality, professional plot style
fig, ax = plt.subplots(figsize=(5, 6), dpi=300)

# Generate the plot using the loaded data
plot_decision_curves(loaded_curves, ax=ax)

# Enhance aesthetics for publication quality
ax.set_title("Decision Curve Analysis (Untouched Test Set)", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Threshold Probability", fontsize=10, fontweight="bold")
ax.set_ylabel("Net Benefit", fontsize=10, fontweight="bold")
ax.set_xlim(0, 0.50)
ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=9)

# Save the high-resolution figure
plot_path = output_dir / "decision_curve_plot.png"
fig.savefig(plot_path, bbox_inches="tight")

plt.tight_layout()
plt.show()
print(f"Plot successfully generated from saved CSV and saved to {plot_path}")